In [ ]:
!pip install pytesseract pillow


In [ ]:
import os
import io
import pandas as pd
from PIL import Image
import pytesseract

# Constants
FOLDER_PATH = "C:/chatbot"
OUTPUT_TEXT_FILE = "extracted_texts.txt"
IMAGE_COLUMN = "image"
IMAGE_BYTES_KEY = "bytes"
TEXT_DELIMITER = "---"


def get_parquet_files(folder_path):
    """Retrieve all .parquet files from a folder."""
    return [os.path.join(folder_path, f) for f in os.listdir(folder_path) if f.endswith(".parquet")]


def extract_text_from_image_bytes(image_bytes):
    """Convert image bytes to text using OCR."""
    try:
        with Image.open(io.BytesIO(image_bytes)) as img:
            return pytesseract.image_to_string(img)
    except Exception as error:
        print(f"Error during OCR: {error}")
        return ""


def process_parquet_file(file_path):
    """Extract texts from a single Parquet file."""
    texts = []
    try:
        df = pd.read_parquet(file_path)
    except Exception as error:
        print(f"Error reading {file_path}: {error}")
        return texts

    for index, row in df.iterrows():
        try:
            img_bytes = row[IMAGE_COLUMN][IMAGE_BYTES_KEY]
            text = extract_text_from_image_bytes(img_bytes)
            if text.strip():
                texts.append(text)
        except KeyError as key_error:
            print(f"Missing key in row {index} of {file_path}: {key_error}")
        except Exception as general_error:
            print(f"Error processing row {index} in {file_path}: {general_error}")

    return texts


def save_texts_to_file(texts, output_file, delimiter):
    """Save a list of texts to a file with a delimiter."""
    try:
        with open(output_file, "w", encoding="utf-8") as f:
            for text in texts:
                f.write(text.strip() + f"\n{delimiter}\n")
        print(f"OCR extraction completed and saved to {output_file}")
    except Exception as error:
        print(f"Error writing to file {output_file}: {error}")


def main():
    all_texts = []
    if not os.path.exists(FOLDER_PATH):
        print(f"Folder not found: {FOLDER_PATH}")
        return

    parquet_files = get_parquet_files(FOLDER_PATH)
    if not parquet_files:
        print("No .parquet files found.")
        return

    for parquet_file in parquet_files:
        print(f"Processing {parquet_file} ...")
        texts = process_parquet_file(parquet_file)
        all_texts.extend(texts)

    if all_texts:
        save_texts_to_file(all_texts, OUTPUT_TEXT_FILE, TEXT_DELIMITER)
    else:
        print("No text was extracted.")


if __name__ == "__main__":
    main()


Processing C:/chatbot\0000.parquet ...
Processing C:/chatbot\0001.parquet ...
Processing C:/chatbot\0002.parquet ...
Processing C:/chatbot\0003.parquet ...
Processing C:/chatbot\0004.parquet ...
Processing C:/chatbot\0005.parquet ...
Processing C:/chatbot\0006.parquet ...
Processing C:/chatbot\0007.parquet ...
Processing C:/chatbot\0008.parquet ...
Processing C:/chatbot\0009.parquet ...
OCR extraction completed and saved to extracted_texts.txt


In [ ]:
import pandas as pd

# Path to your extracted text file
text_file_path = r"C:/Users/choud/Downloads/extracted_texts.txt"
csv_output_path = r"C:/Users/choud/Downloads/extracted_texts.csv"

# Read the entire text file
with open(text_file_path, "r", encoding="utf-8") as file:
    content = file.read()

# Split text into chunks by your delimiter "---"
text_chunks = content.split("\n---\n")

# Remove empty chunks (if any)
text_chunks = [chunk.strip() for chunk in text_chunks if chunk.strip()]

# Create DataFrame
df = pd.DataFrame({"OCR_Text": text_chunks})

# Save to CSV
df.to_csv(csv_output_path, index=False, encoding="utf-8")

print(f"Text data converted to CSV and saved at: {csv_output_path}")


Text data converted to CSV and saved at: C:/Users/choud/Downloads/extracted_texts.csv
